# Data teks besar: baca per potongan, HashingVectorizer + SGD `partial_fit`
**Kapan dipakai:** file train terlalu besar untuk RAM (jutaan baris, beberapa GB) atau TF-IDF
kehabisan memori. Tidak ada kosakata yang disimpan (hashing), model belajar per potongan.

- Validasi tetap: baris dengan `hash(id) % 10 == VAL_BUCKET` jadi validasi (sama tiap baca).
- Beberapa epoch = baca ulang file. Skor validasi dicetak tiap epoch.
- Bobot kelas manual (`class_weight="balanced"` tidak didukung `partial_fit`).
- Test diprediksi per potongan dan ditulis bertahap.

Kalau data muat di RAM, `baseline_oof` (TF-IDF + CV) lebih akurat. Data besar tapi muat:
ambil sampel acak 200–500 ribu baris dulu untuk eksperimen, lalu pakai notebook ini untuk semua.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score

## Konfigurasi

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/data_besar_teks"
OOF_FOLDER = "../../../outputs/text/oof"
DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"   # GANTI saat lomba: sample panitia
KEY_CSV = f"{DATA_FOLDER}/baju_wanita_test_labels.csv"       # GANTI saat lomba (kunci, kalau ada)

ID_COL = "id"                                # GANTI saat lomba
TARGET = "rating"                            # GANTI saat lomba
SUB_COL = "rating"                           # GANTI saat lomba: kolom sample
TEXT_COLS = ["title", "review_text"]         # GANTI saat lomba
CHUNK_ROWS = 4000            # GANTI: 100000–300000 untuk data jutaan baris
N_EPOCHS = 4
N_FEATURES = 2 ** 20         # per vectorizer, 2**21 kalau RAM lega
VAL_BUCKET = 0               # 1 dari 10 ember hash id jadi validasi
BALANCE = True               # bobot kelas = n / (K × jumlah kelas)
SEED = 42


def metric(truth, pred):
    return f1_score(truth, pred, average="macro")      # GANTI: metrik lomba


print("uji metrik:", metric(np.array([0, 1, 1]), np.array([0, 1, 0])))

## Baca ringan: hanya kolom target (cari kelas + bobot)
`usecols` membuat bacaan ini murah walau file besar. Perkiraan RAM penuh dari 1000 baris pertama.

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
labels_only = pd.read_csv(TRAIN_CSV, usecols=[TARGET])
CLASSES, counts = np.unique(labels_only[TARGET].astype(str), return_counts=True)
class_weight = np.ones(len(CLASSES))
if BALANCE:
    class_weight = len(labels_only) / (len(CLASSES) * counts)
head = pd.read_csv(TRAIN_CSV, nrows=1000)
per_row = head.memory_usage(deep=True).sum() / len(head)
print("baris:", len(labels_only), "| kelas:", dict(zip(CLASSES, counts)))
print("perkiraan RAM kalau dibaca penuh:", round(per_row * len(labels_only) / 1e9, 2), "GB")

## Buat sample_submission demo (HAPUS saat lomba)

In [ ]:
test_ids = pd.read_csv(TEST_CSV, usecols=[ID_COL])
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test_ids[ID_COL]
demo_sample[SUB_COL] = CLASSES[0]
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):      # jangan timpa sample panitia
    demo_sample.to_csv(SAMPLE_CSV, index=False)
print("sample demo ditulis:", SAMPLE_CSV, demo_sample.shape)

## Fitur hashing (tanpa fit) + pemisah validasi

In [ ]:
word_hash = HashingVectorizer(n_features=N_FEATURES, ngram_range=(1, 2), alternate_sign=False)
char_hash = HashingVectorizer(n_features=N_FEATURES, analyzer="char_wb", ngram_range=(2, 4),
                              alternate_sign=False)


def featurize(chunk):
    text = chunk[TEXT_COLS].fillna("")
    text = text.astype(str).agg(" . ".join, axis=1)
    return hstack([word_hash.transform(text), char_hash.transform(text)]).tocsr()


def split_chunk(chunk):
    hashed = pd.util.hash_pandas_object(chunk[ID_COL], index=False).to_numpy()
    is_val = hashed % 10 == VAL_BUCKET
    labels = np.searchsorted(CLASSES, chunk[TARGET].astype(str))
    return featurize(chunk), labels, is_val


X, labels, is_val = split_chunk(head)
print("uji potongan:", X.shape, "| porsi validasi:", round(is_val.mean(), 3))

## Latih per potongan, nilai tiap epoch
Isi potongan diacak dulu (file yang terurut per kelas merusak SGD). `average=True` = rata-rata
bobot sepanjang latihan, lebih stabil untuk `partial_fit`.

In [ ]:
model = SGDClassifier(loss="log_loss", alpha=1e-6, average=True, random_state=SEED)
# model = SGDClassifier(loss="modified_huber", alpha=1e-5, average=True)   # alt: margin
rng = np.random.default_rng(SEED)
history = []
START = time.time()
for epoch in range(N_EPOCHS):
    val_truth = []
    val_pred = []
    for chunk in pd.read_csv(TRAIN_CSV, chunksize=CHUNK_ROWS):
        chunk = chunk.iloc[rng.permutation(len(chunk))]
        X, labels, is_val = split_chunk(chunk)
        fit_rows = ~is_val
        model.partial_fit(X[fit_rows], labels[fit_rows], classes=np.arange(len(CLASSES)),
                          sample_weight=class_weight[labels[fit_rows]])
        val_truth.append(labels[is_val])
        val_pred.append(model.predict(X[is_val]))
    score = metric(np.concatenate(val_truth), np.concatenate(val_pred))
    history.append(score)
    print(f"epoch {epoch} | skor validasi {score:.4f} | detik {time.time() - START:.0f}")

Skor validasi epoch 0 dihitung sambil belajar (potongan validasi dinilai setelah potongan latih
sebelumnya), jadi sedikit pesimis. Berhenti di epoch yang skornya berhenti naik.

In [ ]:
figure, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(len(history)), history, marker="o", color="#0072B2")
ax.set_xlabel("epoch")
ax.set_ylabel("skor validasi")
figure.savefig(f"{OUTPUT_FOLDER}/skor_per_epoch.png", dpi=200, bbox_inches="tight")
plt.show()
log_path = f"{OOF_FOLDER}/percobaan.csv"
os.makedirs(OOF_FOLDER, exist_ok=True)
log_row = pd.DataFrame([{"waktu": time.strftime("%Y-%m-%d %H:%M:%S"),
                         "notebook": "data_besar_teks", "model": "hashing_sgd",
                         "metrik": "f1_macro",
                         "skor_cv": round(history[-1], 4), "detik": round(time.time() - START),
                         "catatan": f"validasi hash, {N_EPOCHS} epoch, bukan OOF"}])
log_row.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)

## Tebak test per potongan, tulis bertahap
Hasil sementara ditulis per potongan, lalu disusun ulang mengikuti urutan id sample.

In [ ]:
part_path = f"{OUTPUT_FOLDER}/tebakan_bertahap.csv"
if os.path.exists(part_path):
    os.remove(part_path)
first = True
for chunk in pd.read_csv(TEST_CSV, chunksize=CHUNK_ROWS):
    part = pd.DataFrame()
    part[ID_COL] = chunk[ID_COL].to_numpy()
    part[SUB_COL] = CLASSES[model.predict(featurize(chunk))]
    part.to_csv(part_path, mode="a", header=first, index=False)
    first = False
parts = pd.read_csv(part_path)
print("tebakan test:", parts.shape)

## Submission + cek

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
pred_by_id = pd.Series(parts[SUB_COL].values, index=parts[ID_COL].values)
submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUB_COL] = sample[ID_COL].map(pred_by_id).values
submission.to_csv(f"{OUTPUT_FOLDER}/submission.csv", index=False)
written = pd.read_csv(f"{OUTPUT_FOLDER}/submission.csv")
problems = []
if list(written.columns) != list(sample.columns):
    problems.append(f"kolom/urutan beda: {list(written.columns)} vs {list(sample.columns)}")
if len(written) != len(sample) or written[SUB_COL].isna().sum() > 0:
    problems.append("jumlah baris beda atau ada tebakan kosong")
shares = written[SUB_COL].value_counts(normalize=True)
print(shares.round(3).to_dict())
print("MASALAH:", len(problems), problems)

## Cek dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].astype(str).values, index=key[ID_COL].values)
truth = submission[ID_COL].map(key_by_id)
print("skor test:", round(metric(truth, submission[SUB_COL].astype(str)), 4),
      "| validasi:", round(history[-1], 4))